# Esperimenti manuali — Ottimizzazione Non Vincolata

Notebook per **esperimenti singoli**, configurabili a mano. A differenza di
`experiment.ipynb` (campagna completa), qui si lancia **una sola** combinazione
(metodo, problema, dimensione, modalita' di derivate) sui 6 starting point della
campagna e si producono: riepilogo a schermo, tabella (pandas + LaTeX) e grafici
(convergenza + traiettorie 2D).

**Come si usa:** modifica **solo la Cella 2 (Configurazione)** e poi esegui
tutte le celle in ordine. I 3 metodi disponibili sono Modified Newton (MN),
Truncated Newton (TN) e **Preconditioned Truncated Newton (PTN)**.


## 1 — Import, setup e funzioni helper

Importa i moduli del progetto e **copia** gli helper della campagna
(`experiment.ipynb`): `experimental_rate`, `should_skip`,
`make_sparse_diag_fd_hess`, `build_method_kwargs` (adattato al caso singolo) e
`make_latex_table`. Definisce inoltre l'anagrafica dei problemi, il dispatch dei
metodi e i parametri "best" presi da `fine_tuning.ipynb`.


In [ ]:
# ============================================================
# Import, setup e funzioni helper
# ============================================================
# Gli helper sono COPIATI da experiment.ipynb (fonte della campagna completa) e
# adattati al caso singolo, cosi' i risultati restano confrontabili con le
# tabelle/grafici della campagna.

import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.sparse as sp

# --- Moduli del progetto ---
from src.functions.problem16 import f16, x_bar_16
from src.functions.problem28 import f28, x_bar_28
from src.gradients.problem16 import grad_f16
from src.gradients.problem28 import grad_f28
from src.hessians.problem16 import hess_f16
from src.hessians.problem28 import hess_f28
from src.hessians.finite_diff import hess_fd_diag, hv_fd, hv_fd_normalized_v
from src.methods.modified_newton import modified_newton
from src.methods.truncated_newton import truncated_newton
from src.methods.truncated_newton_preconditioned import preconditioned_truncated_newton
from src.stopping_criteria.absolute.grad_norm import GradNormAbsolute
from src.starting_points import generate_starting_points
from src.finite_diff_factories import make_fd_grad

# --- Anagrafica problemi (id, funzione, gradiente, Hessiana, x_bar) ---
PROBLEMS = {
    'P16': dict(id='P16', f=f16, grad=grad_f16, hess=hess_f16, x_bar_fn=x_bar_16),
    'P28': dict(id='P28', f=f28, grad=grad_f28, hess=hess_f28, x_bar_fn=x_bar_28),
}
X_BAR_FN = {'P16': x_bar_16, 'P28': x_bar_28}

# --- Dispatch metodi e parametri "best" (da experiment.ipynb / fine_tuning) ---
METHOD_FNS = {
    'MN':  modified_newton,
    'TN':  truncated_newton,
    'PTN': preconditioned_truncated_newton,   # stessa firma del TN
}
MN_PARAMS  = dict(alpha0=1.0, c1=1e-4, rho=0.5,
                  beta=1e-2, max_tau_iter=100, max_iter_backtrack=50)
TN_PARAMS  = dict(alpha0=1.0, c1=1e-4, rho=0.9,
                  forcing='quadratic', cg_max_iter=None, max_iter_backtrack=50)
PTN_PARAMS = dict(TN_PARAMS)   # PTN condivide firma e parametri del TN
METHOD_PARAMS = {'MN': MN_PARAMS, 'TN': TN_PARAMS, 'PTN': PTN_PARAMS}

# Soglia memoria per Hessiana densa (come nella campagna).
OOM_THRESHOLD_MB = 4096


def experimental_rate(g_norms, floor=1e-12, window=6):
    """Ordine empirico di convergenza p dalla sequenza ||g_k|| (da experiment.ipynb).

    Modello asintotico  ||g_{k+1}|| ~ C * ||g_k||^p  =>  in scala logaritmica
    p e' la PENDENZA di log||g_{k+1}|| contro log||g_k|| (minimi quadrati sui
    passi in genuina decrescita vicino alla soluzione). Si scartano i valori
    sotto il floor numerico e i passi non decrescenti (che davano rate < 0).
    Ritorna NaN se i punti asintotici non bastano (run troppo corte).
    """
    e = np.asarray(g_norms, dtype=float)
    e = e[np.isfinite(e) & (e > floor)]
    if e.size < 4:
        return float('nan')
    log_e = np.log(e)
    x, y = log_e[:-1], log_e[1:]              # coppie (log||g_k||, log||g_{k+1}||)
    dec = np.where(y < x - 1e-6)[0]           # solo passi in reale decrescita
    if dec.size < 2:
        return float('nan')
    dec = dec[-window:]                        # finestra asintotica
    slope = np.polyfit(x[dec], y[dec], 1)[0]
    return float(slope)


def expected_hessian_mb(n):
    """Memoria (MB) di una Hessiana densa n x n in float64."""
    return n * n * 8 / (1024 ** 2)


def should_skip(method, prob_id, n, deriv_mode):
    """True se la combinazione va saltata per limiti di memoria (cfr. campagna).

    PTN e' trattato come TN: in FD usa il percorso matrix-free (mai skip), in
    exact su P28 fattorizza la Hessiana densa (skip se troppo grande).
    """
    if deriv_mode == 'exact':
        if prob_id == 'P16':
            return False
        return expected_hessian_mb(n) > OOM_THRESHOLD_MB
    if method in ('TN', 'PTN') or prob_id == 'P16':
        return False
    return expected_hessian_mb(n) > OOM_THRESHOLD_MB


def make_sparse_diag_fd_hess(grad_func, k, scaled):
    """Hessiana FD del P16 come matrice sparsa diagonale (memoria O(n))."""
    return lambda x: sp.diags(
        hess_fd_diag(grad_func, x, k=k, scaled=scaled), 0, format='csc')


def build_method_kwargs(method_name, method_params, pinfo, dcfg, hv_func=hv_fd):
    """Costruisce i kwargs per la chiamata al metodo (adattato da experiment.ipynb).

    - exact: derivate analitiche (in exact PTN puo' precondizionare con la
      Hessiana assemblata: diagonale per P16, densa per P28).
    - FD: solo MN+P16 riceve la Hessiana FD DIAGONALE SPARSA (memoria O(n));
      TN/PTN lasciano hess_f=None (matrix-free) -> PTN ricade su CG NON
      precondizionato (warning una tantum).
    - hv_func (solo TN/PTN): sceglie hv_fd vs hv_fd_normalized_v sul percorso
      matrix-free. Inerte in exact (la Hessiana c'e').
    """
    kw = dict(method_params)
    use_p16_sparse_hess = (method_name == 'MN' and pinfo.get('id') == 'P16')

    if dcfg['deriv_mode'] == 'exact':
        kw['grad_f'] = pinfo['grad']
        kw['hess_f'] = pinfo['hess']
    elif dcfg['deriv_mode'] == 'only_hess_fd':
        grad_callable = pinfo['grad']
        kw['grad_f'] = grad_callable
        kw['hess_f'] = (make_sparse_diag_fd_hess(grad_callable, dcfg['k'], dcfg['scaled'])
                        if use_p16_sparse_hess else None)
        kw['k'] = dcfg['k']
        kw['scaled'] = dcfg['scaled']
    elif dcfg['deriv_mode'] == 'both_fd':
        if use_p16_sparse_hess:
            grad_callable = make_fd_grad(pinfo['f'], k=dcfg['k'], scaled=dcfg['scaled'])
            kw['grad_f'] = grad_callable
            kw['hess_f'] = make_sparse_diag_fd_hess(grad_callable, dcfg['k'], dcfg['scaled'])
        else:
            kw['grad_f'] = None
            kw['hess_f'] = None
        kw['k'] = dcfg['k']
        kw['scaled'] = dcfg['scaled']
    elif dcfg['deriv_mode'] == 'both_fd_special':
        grad_callable = make_fd_grad(pinfo['f'], k=dcfg['k_grad'], scaled=dcfg['scaled'])
        kw['grad_f'] = grad_callable
        kw['hess_f'] = (make_sparse_diag_fd_hess(grad_callable, dcfg['k_hess'], dcfg['scaled'])
                        if use_p16_sparse_hess else None)
        kw['k'] = dcfg['k_hess']
        kw['scaled'] = dcfg['scaled']

    if method_name in ('TN', 'PTN'):
        kw['hv_func'] = hv_func   # MN non accetta hv_func
    return kw


def _tex_escape(s):
    """Escape underscores for LaTeX text mode (da experiment.ipynb)."""
    return str(s).replace("_", r"\_")


def make_latex_table(gdf, method, prob, n, deriv_mode, h_type, k_val):
    """Genera il codice LaTeX per una singola tabella (da experiment.ipynb)."""
    gdf = gdf.sort_values("start_pt_id")

    cap_parts = [
        _tex_escape(method),
        _tex_escape(prob),
        f"$n={n}$",
        r"\texttt{" + _tex_escape(deriv_mode) + "}",
    ]

    if deriv_mode == "both_fd_special":
        cap_parts += [
            f"$h={h_type}$",
            r"$k_{\mathrm{grad}}=8$",
            r"$k_{\mathrm{hess}}=4$",
        ]
    elif deriv_mode != "exact":
        cap_parts += [f"$h={h_type}$", f"$k={k_val}$"]

    caption = ", ".join(cap_parts)

    lines = []
    lines.append(r"\begin{table}[htbp]")
    lines.append(r"\centering")
    lines.append(r"\begin{tabular}{cccclcr}")
    lines.append(r"\toprule")
    lines.append(
        r"SP & $\|\nabla f\|$ & iter/max & success & flag & conv\_rate & time (s) \\"
    )
    lines.append(r"\midrule")

    for _, row in gdf.iterrows():
        sp_id = int(row["start_pt_id"])
        gn = f"${row['grad_norm']:.2e}$" if not np.isnan(row["grad_norm"]) else "---"
        it_max = f"{int(row['iters'])}/{int(row['max_iters'])}"
        suc = "yes" if row["success"] else "no"
        flag = r"\texttt{" + _tex_escape(row["flag"]) + "}"
        cr = f"{row['conv_rate']:.2f}" if not np.isnan(row["conv_rate"]) else "---"
        tm = f"{row['time']:.2f}"
        lines.append(f"{sp_id} & {gn} & {it_max} & {suc} & {flag} & {cr} & {tm} \\\\")

    lines.append(r"\midrule")

    succ = gdf[gdf["success"] == True]
    if len(succ) > 0:
        avg_gn = f"${succ['grad_norm'].mean():.2e}$"
        avg_it = f"{succ['iters'].mean():.1f}/{int(succ['max_iters'].iloc[0])}"
        avg_cr = succ["conv_rate"].dropna()
        avg_cr_str = f"{avg_cr.mean():.2f}" if len(avg_cr) > 0 else "---"
        avg_tm = f"{succ['time'].mean():.2f}"
        lines.append(
            f"Avg & {avg_gn} & {avg_it} & {len(succ)}/{len(gdf)} & --- & {avg_cr_str} & {avg_tm} \\\\"
        )
    else:
        lines.append(f"Avg & --- & --- & 0/{len(gdf)} & --- & --- & --- \\\\")

    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}")
    lines.append(r"\caption{" + caption + "}")
    lines.append(r"\end{table}")

    return "\n".join(lines)


def param_tag():
    """Stringa che riassume la configurazione, per i nomi dei file salvati.

    Legge le variabili globali della Cella 2 (risolte a tempo di chiamata).
    """
    parts = [METHOD, PROBLEM, f"n{N}", DERIV_MODE]
    if DERIV_MODE == 'both_fd_special':
        parts += [H_TYPE, f"kg{K_GRAD}", f"kh{K_HESS}"]
    elif DERIV_MODE != 'exact':
        parts += [H_TYPE, f"k{K}"]
    if USE_NORMALIZED_HV and METHOD in ('TN', 'PTN'):
        parts.append("normhv")
    return "_".join(parts)


print("Setup completato. Metodi:", list(METHOD_FNS), "| Problemi:", list(PROBLEMS))


## 2 — Configurazione (UNICA cella da modificare)

Imposta metodo, problema, dimensione, modalita' di derivate, parametri delle
differenze finite, versione di `hv_func` e criteri di stop. I valori di default
coincidono con quelli della campagna (`tol=1e-8`, `MAX_ITER=5000`,
`TIME_LIMIT=20`), cosi' i risultati sono confrontabili con le tabelle complete.


In [ ]:
# -- Metodo -------------------------------------------------------
METHOD        = "TN"          # "TN"  = Truncated Newton
                              # "PTN" = Preconditioned Truncated Newton
                              # "MN"  = Modified Newton

# -- Problema -----------------------------------------------------
PROBLEM       = "P16"         # "P16" (Banded Trigonometric, Hessiana diagonale)
                              # "P28" (Variably Dimensioned, Hessiana densa)

# -- Dimensione ---------------------------------------------------
N             = 2             # 2, 1000, 10000, 100000

# -- Modalita' derivate -------------------------------------------
DERIV_MODE    = "exact"       # "exact"           -> derivate analitiche
                              # "only_hess_fd"    -> grad esatto, Hess/Hv via FD
                              # "both_fd"          -> grad e Hess/Hv via FD (stesso k)
                              # "both_fd_special"  -> grad e Hess con k diversi

# -- Parametri differenze finite ----------------------------------
H_TYPE        = "fixed"       # "fixed"    -> h   = 10^(-k)            (scaled=False)
                              # "specific" -> h_i = 10^(-k)*|x_i|       (scaled=True)
                              #   (per gli Hv: h = 10^(-k)*max(||x||_inf, 1))
K             = 4             # k in {4, 8, 12}; usato per grad e Hess,
                              #   IGNORATO se DERIV_MODE = "both_fd_special"
K_GRAD        = 8             # k del gradiente (solo "both_fd_special"; canonico 8)
K_HESS        = 4             # k della Hessiana (solo "both_fd_special"; canonico 4)

# -- Versione hv_func (solo TN/PTN su percorso matrix-free) -------
USE_NORMALIZED_HV = False     # False -> hv_fd standard
                              # True  -> hv_fd_normalized_v
                              # Rilevante solo quando hess_f=None (only_hess_fd /
                              # both_fd / both_fd_special per TN/PTN). Ignorato da
                              # MN e in modalita' "exact".

# -- Criteri di stop (default = valori della campagna) ------------
GRAD_TOL      = 1e-8          # GradNormAbsolute(tol=GRAD_TOL)  (campagna: 1e-8)
MAX_ITER      = 5000          # iterazioni massime              (campagna: 5000)
TIME_LIMIT    = 20            # secondi, None = nessun limite   (campagna: 20)

print(f"METHOD={METHOD} | PROBLEM={PROBLEM} | N={N} | DERIV_MODE={DERIV_MODE}")
print(f"H_TYPE={H_TYPE} | K={K} | K_GRAD={K_GRAD} | K_HESS={K_HESS}")
print(f"USE_NORMALIZED_HV={USE_NORMALIZED_HV} | GRAD_TOL={GRAD_TOL} | "
      f"MAX_ITER={MAX_ITER} | TIME_LIMIT={TIME_LIMIT}")


## 3 — Starting point identici alla campagna

Riproduce **esattamente** i 6 punti usati da `experiment.ipynb` per
`(PROBLEM, N)`. La campagna semina la RNG **una sola volta** e poi estrae i punti
in un **ordine fisso** (problemi x dimensioni): poiche' la `rng` e' condivisa,
per ottenere gli stessi numeri bisogna rigenerare l'intera cache nello stesso
ordine. Saltare problemi/dimensioni cambierebbe le estrazioni.


In [ ]:
# ============================================================
# Starting point IDENTICI alla campagna (experiment.ipynb)
# ============================================================
SEED = 323334
CAMPAIGN_DIMS = (2, 1000, 10000, 100000)   # stesse dimensioni della campagna

rng = np.random.default_rng(SEED)
starts_cache = {}
for prob_id in ('P16', 'P28'):              # ordine fisso problemi
    for n in CAMPAIGN_DIMS:                  # ordine fisso dimensioni
        x_bar = X_BAR_FN[prob_id](n)
        starts_cache[(prob_id, n)] = generate_starting_points(
            x_bar, num_random=5, rng=rng)    # x_bar + 5 random in [x_bar-1, x_bar+1]

if (PROBLEM, N) in starts_cache:
    all_x0 = starts_cache[(PROBLEM, N)]      # 6 punti identici alla campagna
    print(f"Starting point per ({PROBLEM}, n={N}): {len(all_x0)} "
          f"(identici alla campagna, SEED={SEED})")
else:
    # N non canonico: impossibile riprodurre la campagna -> punti nuovi.
    print(f"ATTENZIONE: N={N} non e' fra {CAMPAIGN_DIMS}; i punti NON "
          f"corrispondono alla campagna. Ne genero di nuovi (SEED={SEED}).")
    all_x0 = generate_starting_points(
        X_BAR_FN[PROBLEM](N), num_random=5, rng=np.random.default_rng(SEED))

# Stampa i 6 punti (o solo le norme se N e' grande).
for si, x0 in enumerate(all_x0):
    tag = " (x_bar)" if si == 0 else ""
    if N <= 10:
        print(f"  sp={si}{tag}: x0 = {np.asarray(x0)}")
    else:
        print(f"  sp={si}{tag}: ||x0|| = {np.linalg.norm(x0):.4e}")


## 4 — Esecuzione sui 6 starting point

Costruisce la configurazione delle derivate (`dcfg`) dai valori scalari della
Cella 2, verifica i limiti di memoria con `should_skip`, poi lancia il metodo
scelto su ogni punto salvando la storia di `||grad||` (e il path se `N==2`).
Stampa un riepilogo dal vivo per ciascuna run.


In [ ]:
# ============================================================
# Esecuzione del metodo sui 6 starting point
# ============================================================
scaled = (H_TYPE == 'specific')
if DERIV_MODE == 'exact':
    dcfg = dict(deriv_mode='exact', h_type='', k=None, scaled=None)
elif DERIV_MODE == 'both_fd_special':
    dcfg = dict(deriv_mode='both_fd_special', h_type=H_TYPE, k='g8h4',
                k_grad=K_GRAD, k_hess=K_HESS, scaled=scaled)
else:
    dcfg = dict(deriv_mode=DERIV_MODE, h_type=H_TYPE, k=K, scaled=scaled)

method_fn = METHOD_FNS[METHOD]
pinfo = PROBLEMS[PROBLEM]
hv_choice = hv_fd_normalized_v if USE_NORMALIZED_HV else hv_fd

rows = []     # una riga per starting point (per la tabella)
runs = []     # storia per i grafici

if should_skip(METHOD, PROBLEM, N, DERIV_MODE):
    print(f"SKIP: ({METHOD}, {PROBLEM}, n={N}, {DERIV_MODE}) supera la soglia di "
          f"memoria ({OOM_THRESHOLD_MB} MB per Hessiana densa). "
          f"Cambia configurazione nella Cella 2.")
else:
    mkw = build_method_kwargs(METHOD, METHOD_PARAMS[METHOD], pinfo, dcfg,
                              hv_func=hv_choice)
    for si, x0 in enumerate(all_x0):
        stop = GradNormAbsolute(tol=GRAD_TOL)
        t0 = time.perf_counter()
        res = method_fn(pinfo['f'], x0, stop,
                        max_iter=MAX_ITER, time_limit=TIME_LIMIT,
                        return_history=True, **mkw)
        elapsed = time.perf_counter() - t0

        hist = res.get('history', [])
        g_norms = [h['grad_norm'] for h in hist]
        x_path = [h['x'] for h in hist] if N == 2 else None
        rate = experimental_rate(g_norms) if res['success'] else float('nan')

        rows.append(dict(
            start_pt_id=si, grad_norm=res['grad_norm'], iters=res['n_iter'],
            max_iters=MAX_ITER, success=res['success'], flag=res['stop_reason'],
            conv_rate=rate, time=elapsed,
        ))
        runs.append(dict(start_pt_id=si, g_norms=g_norms, x_path=x_path,
                         success=res['success'], conv_rate=rate))

        tag = " (x_bar)" if si == 0 else ""
        cr_str = f"{rate:.2f}" if np.isfinite(rate) else "---"
        print("-" * 44)
        print(f"Starting point {si}{tag}")
        print("-" * 44)
        print(f"  Iterations   : {res['n_iter']} / {MAX_ITER}")
        print(f"  Grad norm    : {res['grad_norm']:.2e}")
        print(f"  Success      : {'YES' if res['success'] else 'NO'}")
        print(f"  Flag         : {res['stop_reason']}")
        print(f"  Conv. rate   : {cr_str}")
        print(f"  Time         : {elapsed:.2f}s")


## 5 — Tabella riassuntiva + export LaTeX

DataFrame con una riga per starting point e una riga finale `Avg` calcolata
**solo sulle run riuscite**. La tabella LaTeX e' generata riusando
`make_latex_table` di `experiment.ipynb` e salvata in `results/tables/` con un
nome che codifica tutti i parametri (suffisso `_normhv` se `USE_NORMALIZED_HV`).


In [ ]:
# ============================================================
# Tabella riassuntiva (pandas) + export LaTeX
# ============================================================
if not rows:
    print("Nessun risultato (run saltata): esegui prima la Cella 4 con una "
          "configurazione valida.")
else:
    df = pd.DataFrame(rows)

    # --- DataFrame di visualizzazione con riga Avg (solo successi) ---
    succ = df[df['success'] == True]
    df_show = df.copy()
    df_show['iters'] = df_show['iters'].astype(int)
    if len(succ) > 0:
        avg = {
            'start_pt_id': 'Avg',
            'grad_norm': succ['grad_norm'].mean(),
            'iters': round(succ['iters'].mean(), 1),
            'max_iters': MAX_ITER,
            'success': f"{len(succ)}/{len(df)}",
            'flag': '',
            'conv_rate': succ['conv_rate'].dropna().mean(),
            'time': succ['time'].mean(),
        }
    else:
        avg = {'start_pt_id': 'Avg', 'grad_norm': np.nan, 'iters': np.nan,
               'max_iters': MAX_ITER, 'success': f"0/{len(df)}", 'flag': '',
               'conv_rate': np.nan, 'time': np.nan}
    df_show = pd.concat([df_show, pd.DataFrame([avg])], ignore_index=True)

    header = (f"Metodo={METHOD}  Problema={PROBLEM}  n={N}  deriv={DERIV_MODE}"
              + (f"  h={H_TYPE}" if DERIV_MODE != 'exact' else "")
              + (f"  normhv={USE_NORMALIZED_HV}" if METHOD in ('TN', 'PTN') else ""))
    print(header)
    display(df_show)

    # --- Export LaTeX (riusa make_latex_table) ---
    tables_dir = Path('results') / 'tables'
    tables_dir.mkdir(parents=True, exist_ok=True)
    k_for_caption = K if DERIV_MODE not in ('exact', 'both_fd_special') else ''
    h_for_caption = H_TYPE if DERIV_MODE != 'exact' else ''
    tex = make_latex_table(df, METHOD, PROBLEM, N, DERIV_MODE,
                           h_for_caption, k_for_caption)
    tex_path = tables_dir / f"table_{param_tag()}.tex"
    tex_path.write_text(tex, encoding='utf-8')
    print(f"\nLaTeX salvato in: {tex_path}\n")
    print(tex)


## 6 — Grafici

**Grafico 1 (sempre):** convergenza `||grad||` vs iterazione in `semilogy`
(asse y logaritmico, asse x lineare = numero di iterazione) per ogni starting
point riuscito. *Nota:* la campagna `experiment.ipynb` usa invece `loglog`; qui
si segue la richiesta `semilogy`.

**Grafico 2 (solo se `N==2`):** contour della funzione obiettivo con le
traiettorie delle iterate sovrapposte per tutti e 6 gli starting point
(range P16 `[-2pi, 2pi]`, P28 `[-1, 3]` con clipping `min(Z, 30)`).
I file PNG (dpi=200) vanno in `results/plots/`.


In [ ]:
# ============================================================
# Grafici: convergenza (semilogy) + traiettorie 2D (solo N==2)
# ============================================================
if not runs:
    print("Nessun risultato da plottare (run saltata).")
else:
    plots_dir = Path('results') / 'plots'
    plots_dir.mkdir(parents=True, exist_ok=True)
    colors = plt.cm.tab10(np.linspace(0, 1, 10))

    title_cfg = f"{METHOD} | {PROBLEM} | n={N} | {DERIV_MODE}"
    if DERIV_MODE != 'exact':
        title_cfg += f" | h={H_TYPE}"
        title_cfg += (f" | kg{K_GRAD},kh{K_HESS}" if DERIV_MODE == 'both_fd_special'
                      else f" | k={K}")
    if USE_NORMALIZED_HV and METHOD in ('TN', 'PTN'):
        title_cfg += " | normhv"

    # --- Grafico 1: convergenza (semilogy: y log, x = iterazione) ---
    fig, ax = plt.subplots(figsize=(10, 6))
    any_plot = False
    for r in runs:
        if not r['success']:
            continue
        g = r['g_norms']
        if len(g) < 2:
            continue
        iters = np.arange(1, len(g) + 1)
        cr = r['conv_rate']
        lbl = (f"sp={r['start_pt_id']} (rate={cr:.2f})" if np.isfinite(cr)
               else f"sp={r['start_pt_id']}")
        ax.semilogy(iters, g, 'o-', color=colors[r['start_pt_id']],
                    markersize=3, linewidth=1.2, label=lbl)
        any_plot = True

    if any_plot:
        ax.set_xlabel('Iterazione')
        ax.set_ylabel(r'$\|\nabla f(x_k)\|$')
        ax.set_title(f"Convergenza - {title_cfg}")
        ax.legend(fontsize=8)
        ax.grid(True, which='both', alpha=0.3)
        plt.tight_layout()
        conv_path = plots_dir / f"conv_{param_tag()}.png"
        fig.savefig(conv_path, dpi=200, bbox_inches='tight')
        plt.show()
        print(f"Grafico convergenza salvato in: {conv_path}")
    else:
        plt.close(fig)
        print("Nessuna run riuscita: grafico di convergenza omesso.")

    # --- Grafico 2: contour + traiettorie (solo N==2) ---
    if N != 2:
        print(f"N={N} != 2: contour plot omesso "
              f"(la funzione vive in R^{N}, non rappresentabile in 2D).")
    else:
        lo, hi = (-2 * np.pi, 2 * np.pi) if PROBLEM == 'P16' else (-1.0, 3.0)
        ng = 300
        x1r = np.linspace(lo, hi, ng)
        x2r = np.linspace(lo, hi, ng)
        X1, X2 = np.meshgrid(x1r, x2r)
        Z = np.vectorize(lambda a, b: pinfo['f'](np.array([a, b])))(X1, X2)
        if PROBLEM == 'P28':
            Z = np.minimum(Z, 30)   # clipping per leggibilita' (cfr. campagna)
        cmap = 'viridis' if PROBLEM == 'P16' else 'plasma'

        fig, ax = plt.subplots(figsize=(8, 7))
        cf = ax.contourf(X1, X2, Z, levels=40, cmap=cmap)
        ax.contour(X1, X2, Z, levels=40, colors='white', linewidths=0.3,
                   linestyles='dashed', alpha=0.4)
        fig.colorbar(cf, ax=ax, fraction=0.046, pad=0.04).set_label('$F(x)$')

        x_bar = pinfo['x_bar_fn'](2)
        ax.plot(x_bar[0], x_bar[1], 'o', color='red', markersize=10, zorder=20,
                label=f"$\\bar{{x}}$ = ({x_bar[0]:.1f}, {x_bar[1]:.1f})")

        for r in runs:
            if r['x_path'] is None or len(r['x_path']) == 0:
                continue
            path = np.array(r['x_path'])
            si = r['start_pt_id']
            lbl = f"sp={si}" + (r" ($\bar{x}$)" if si == 0 else "")
            ax.plot(path[:, 0], path[:, 1], '-', color=colors[si],
                    linewidth=1.8, zorder=15, label=lbl)
            ax.scatter(path[:, 0], path[:, 1], color=colors[si], s=15,
                       zorder=16, edgecolors='black', linewidths=0.3)
            ax.scatter([path[-1, 0]], [path[-1, 1]], color='cyan', s=120,
                       marker='*', edgecolors='black', linewidths=0.5, zorder=18)

        ax.set_xlim(lo, hi)
        ax.set_ylim(lo, hi)
        ax.set_aspect('equal')
        ax.set_xlabel('$x_1$')
        ax.set_ylabel('$x_2$')
        ax.set_title(f"Traiettorie - {title_cfg}")
        ax.legend(fontsize=8, loc='upper right', framealpha=0.9)
        plt.tight_layout()
        cont_path = plots_dir / f"contour_{param_tag()}.png"
        fig.savefig(cont_path, dpi=200, bbox_inches='tight')
        plt.show()
        print(f"Contour plot salvato in: {cont_path}")
